# Day 4: Ensuring Agent Quality with Observability and Evaluation

Welcome to Day 4! So far, you've learned how to build and add memory to AI agents. But building an agent is just the first step. How do you know if it's working correctly, efficiently, and reliably? How do you fix it when it breaks in mysterious ways?

Today, we'll dive into **Agent Quality**, which rests on two essential pillars:

1.  **🔎 Observability**: The ability to peek under the hood and see *exactly* what your agent is doing, thinking, and saying at every step. This is your primary tool for debugging.

2.  **📝 Evaluation**: The process of systematically testing your agent against a set of standards to measure its performance. This is how you proactively catch issues and ensure quality over time.

By the end of this notebook, you'll know how to monitor your agent's internal workings and create automated tests to grade its performance.


## ⚙️ Setup

First, let's install the `google-adk` library and configure our API key. This notebook is designed for Google Colab.


In [ ]:
# Install the google-adk library
!pip install -q google-adk

import os
import asyncio
from google.colab import userdata

# Get your API key from Colab secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    print('✅ GOOGLE_API_KEY configured successfully.')
except Exception as e:
    print('🔑 ERROR: Please add your GOOGLE_API_KEY to Colab Secrets. Go to the key icon on the left sidebar.')


## Part 1: Observability - Peeking Under the Hood

### What is Agent Observability?

Unlike traditional software that often fails with clear error messages, AI agents can fail silently or behave in unexpected ways. 

**The Challenge:**
```
User: "Find me some good Italian restaurants nearby."
Agent: "I cannot help with that request."
You: 😭 WHY?? Is the prompt bad? Is a tool missing? Did an API fail?
```

**The Solution:**
Observability gives you a window into the agent's mind. It lets you trace the entire decision-making process: the prompts sent to the LLM, the tools it considered, the tools it chose to call, and the data that flowed between each step. 

ADK comes with a built-in `LoggingPlugin` that makes this incredibly easy. It hooks into the agent's lifecycle and prints a detailed log of every major event.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.google_search_tool import google_search
from google.adk.runners import InMemoryRunner
from google.adk.plugins.logging_plugin import LoggingPlugin

# 1. Define a simple agent that uses a tool
simple_search_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="You are a helpful search assistant. Use the google_search tool to answer the user's question.",
    tools=[google_search]
)

# 2. Create a runner and attach the LoggingPlugin
# This plugin will automatically print detailed logs for every step.
runner = InMemoryRunner(
    agent=simple_search_agent,
    plugins=[LoggingPlugin()]
)

# 3. Run the agent with a query
# The run_debug() method prints a user-friendly summary.
# Watch the output below to see the full trace, including the tool call!
async def run_search():
    print("🚀 Running agent with LoggingPlugin...\n")
    response = await runner.run_debug("What is the latest news about generative AI?")
    print("\n--- Agent's Final Response ---")
    print(response)

# In a notebook, we use asyncio.run() to execute our async function
asyncio.run(run_search())


## Part 2: Evaluation - Grading Your Agent's Performance

Observability is great for debugging a *single* run, but how do you ensure your agent performs well across *hundreds* of different scenarios? Manual testing is too slow and doesn't scale.

This is where **Evaluation** comes in. The goal is to create an automated system to test and grade your agent's responses.

### LLM-as-a-Judge

A powerful and modern technique for evaluation is called **LLM-as-a-Judge**. The concept is simple:

1.  **Primary Agent**: The agent we want to test (e.g., a calculator, a research assistant).
2.  **Test Dataset**: A list of questions and their known correct answers.
3.  **Evaluator Agent (The Judge)**: A *second* LLM agent whose only job is to compare the Primary Agent's answer to the correct answer and give a grade (e.g., "Pass" or "Fail").

This approach is powerful because the Judge can understand nuance, context, and variations in language, making it much more flexible than simple string matching.


In [ ]:
# Let's build a simple calculator agent to test.

# 1. Define the tool and the primary agent
def add(a: int, b: int) -> int:
    """Adds two integers together."""
    return a + b

calculator_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="You are a calculator. Use the 'add' tool to answer the user's math question.",
    tools=[add]
)

calculator_runner = InMemoryRunner(agent=calculator_agent)

# 2. Create a small test dataset with questions and expected answers
test_dataset = [
    {'query': 'What is 2 + 2?', 'expected_answer': '4'},
    {'query': 'Can you add 5 and 10 for me?', 'expected_answer': '15'},
    {'query': 'what is 99 plus 1', 'expected_answer': '100'},
    {'query': 'what is seven plus three', 'expected_answer': '10'} # A slightly harder one!
]

# 3. Run the agent against the test dataset
async def run_tests():
    results = []
    print("Running tests on Calculator Agent...\n")
    for i, test_case in enumerate(test_dataset):
        print(f"--- Test Case {i+1} ---")
        query = test_case['query']
        print(f"Query: {query}")
        
        # Get the agent's actual response
        response = await calculator_runner.run(query)
        actual_answer = response['output']
        print(f"Agent Response: {actual_answer}")
        
        results.append({
            'query': query,
            'expected_answer': test_case['expected_answer'],
            'actual_answer': actual_answer
        })
        print("--------------------\n")
    return results

# Run the tests and store the raw results
test_results = asyncio.run(run_tests())


In [ ]:
# Now, let's create our LLM-as-a-Judge to grade the results.

# 1. Define the Evaluator Agent (The Judge)
evaluator_instruction = """
You are a grading assistant. You will be given an 'expected_answer' and an 'actual_answer' from a calculator agent.
Your task is to determine if the numerical result in the 'actual_answer' correctly matches the 'expected_answer'.
The 'actual_answer' might contain extra conversational text, but the core number must be correct.
Respond with only the word "Pass" or "Fail".
"""

evaluator_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction=evaluator_instruction
)

evaluator_runner = InMemoryRunner(agent=evaluator_agent)

# 2. Grade each result from the previous step
async def run_evaluation():
    print("⚖️  Running LLM-as-a-Judge evaluation...\n")
    graded_results = []
    for result in test_results:
        # Create a prompt for the judge
        eval_prompt = f"Expected Answer: {result['expected_answer']}\nActual Answer: {result['actual_answer']}\n\nGrade:"
        
        # Get the grade from the evaluator agent
        grade_response = await evaluator_runner.run(eval_prompt)
        grade = grade_response['output'].strip()
        
        graded_results.append({
            'query': result['query'],
            'actual_answer': result['actual_answer'],
            'grade': grade
        })
    return graded_results

# Run the evaluation
final_grades = asyncio.run(run_evaluation())

# 3. Display the final report
import pandas as pd
df = pd.DataFrame(final_grades)
print("📊 Evaluation Report")
print(df.to_string())


## 🚀 Mini-Capstone Update: Evaluating Your Research Assistant

In Day 3, you built a Personal Research Assistant and might have instructed it to use a specific output format (like a bulleted list or a JSON object).

But how do you *guarantee* it follows those formatting rules every single time? Manually checking each response is not a scalable solution.

Now, you'll apply today's LLM-as-a-Judge concept to your capstone project. We will create a specialized **'Formatting Judge'** agent to automatically verify that your research assistant is adhering to its formatting instructions.


In [ ]:
# Goal: Ensure our Research Assistant always uses a bulleted list.

# 1. Define the Personal Research Assistant (with formatting instructions)
research_assistant_instruction = """
You are a helpful research assistant. 
Use the google_search tool to find information on the user's topic. 
Summarize the key findings and present them in a bulleted list. Each point should start with a '*' character.
"""

research_assistant = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction=research_assistant_instruction,
    tools=[google_search]
)

research_runner = InMemoryRunner(agent=research_assistant)

# 2. Define the Formatting Evaluator (The Formatting Judge)
formatting_evaluator_instruction = """
You are a strict formatting evaluator. You will be given a piece of text.
Your only job is to check if the text contains a bulleted list, where lines start with a '*' character.
If it contains at least one line starting with '*', respond with only the word "Pass".
Otherwise, respond with only the word "Fail".
"""

formatting_evaluator = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction=formatting_evaluator_instruction
)

formatting_runner = InMemoryRunner(agent=formatting_evaluator)

# 3. Run the test and evaluation
async def run_capstone_test():
    test_query = "What are the main benefits of using Python for data science?"
    print(f"Test Query: {test_query}\n")
    
    # Get the research assistant's response
    print("--- Research Assistant's Response ---")
    assistant_response = await research_runner.run(test_query)
    assistant_output = assistant_response['output']
    print(assistant_output)
    print("-------------------------------------\n")
    
    # Get the formatting judge's grade
    print("--- Formatting Judge's Verdict ---")
    grade_response = await formatting_runner.run(assistant_output)
    grade = grade_response['output'].strip()
    print(f"Formatting Grade: {grade}")
    print("----------------------------------")

# Run the capstone evaluation test
asyncio.run(run_capstone_test())
